In [1]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.models import Model

In [2]:
# English → French
eng_sentences = [
    "i eat apples",
    "i like dogs"
]

fra_sentences = [
    "je mange des pommes",
    "j aime les chiens"
]

fra_sentences = [f"<start> {s} <end>" for s in fra_sentences] #adds <start> & end to fra sentences


In [3]:
# Encoder tokenizer (English)
enc_tokenizer = Tokenizer()
enc_tokenizer.fit_on_texts(eng_sentences)

# Decoder tokenizer (French)
dec_tokenizer = Tokenizer(filters="") #filters to presserve <> for start and stop
dec_tokenizer.fit_on_texts(fra_sentences)

In [4]:
encoder_input = enc_tokenizer.texts_to_sequences(eng_sentences)
decoder_input = dec_tokenizer.texts_to_sequences(fra_sentences)

# Shift decoder target (teacher forcing - sending the actual word rather than the predicted word)
decoder_target = [seq[1:] for seq in decoder_input]
decoder_input  = [seq[:-1] for seq in decoder_input]


In [5]:
# Pad encoder and decoder sequences so all sentences have the same length
# (required for batching and tensor operations in RNNs)

max_enc_len = max(len(seq) for seq in encoder_input)
max_dec_len = max(len(seq) for seq in decoder_input)

encoder_input = pad_sequences(encoder_input, maxlen=max_enc_len, padding="post")
decoder_input = pad_sequences(decoder_input, maxlen=max_dec_len, padding="post")
decoder_target = pad_sequences(decoder_target, maxlen=max_dec_len, padding="post")


In [6]:
embedding_dim = 32 #Each word will be represented as a 32-dimensional vector.
latent_dim = 64 #size of lstm hidden state 

encoder_inputs = Input(shape=(None,), name="encoder_inputs")

encoder_embedding = Embedding(
    input_dim=len(enc_tokenizer.word_index) + 1, #input size for embedding layer
    output_dim=embedding_dim, #output vector of size 32
    name="encoder_embedding" 
)

enc_embed = encoder_embedding(encoder_inputs) #passing inputs to embedding layer

encoder_lstm = LSTM(
    latent_dim,
    return_state=True,
    name="encoder_lstm"
)

_, state_h, state_c = encoder_lstm(enc_embed)
encoder_states = [state_h, state_c]


#Decoder - Embeddings

decoder_inputs = Input(shape=(None,), name="decoder_inputs")

decoder_embedding = Embedding(
    input_dim=len(dec_tokenizer.word_index) + 1,
    output_dim=embedding_dim,
    name="decoder_embedding"
)

dec_embed = decoder_embedding(decoder_inputs)

decoder_lstm = LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True,
    name="decoder_lstm"
)

decoder_outputs, _, _ = decoder_lstm(
    dec_embed,
    initial_state=encoder_states
)

decoder_dense = Dense(
    len(dec_tokenizer.word_index) + 1,
    activation="softmax",
    name="decoder_dense"
)

decoder_outputs = decoder_dense(decoder_outputs)

#Training the model. 

model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy"
)

model.fit(
    [encoder_input, decoder_input],
    decoder_target,
    epochs=300,
    verbose=0
)


In [39]:
model.summary()


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, None, 32)  │        192 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, None, 32)  │        352 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 64),      │     24,832 │ encoder_embeddin… │
│                     │ (None, 64),       │            │                   │
│                     │ (None, 64)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, None,     │     24,832 │ decoder_embeddin… │
│                     │ 64), (None, 64),  │            │ encoder_lstm[0][… │
│                     │ (None, 64)]       │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_dense       │ (None, None, 11)  │        715 │ decoder_lstm[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 152,771 (596.77 KB)

 Trainable params: 50,923 (198.92 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 101,848 (397.85 KB)

In [15]:
dec_tokenizer.word_index

{'<start>': 1,
 '<end>': 2,
 'je': 3,
 'mange': 4,
 'des': 5,
 'pommes': 6,
 'j': 7,
 'aime': 8,
 'les': 9,
 'chiens': 10}

In [9]:
#Encoder Inference Model

encoder_model = Model(
    encoder_inputs,
    encoder_states
)

#Decoder Inference Model

decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

# Reuse SAME embedding layer
dec_embed_inf = decoder_embedding(decoder_inputs)

decoder_outputs_inf, state_h_inf, state_c_inf = decoder_lstm(
    dec_embed_inf,
    initial_state=decoder_states_inputs
)

decoder_states_inf = [state_h_inf, state_c_inf]

# Reuse SAME Dense layer
decoder_outputs_inf = decoder_dense(decoder_outputs_inf)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs_inf] + decoder_states_inf
)


In [ ]:
#Encoder Embeddings

encoder_embedding_layer = model.get_layer("embedding_2")
encoder_embedding_matrix = encoder_embedding_layer.get_weights()[0]

for word, idx in enc_tokenizer.word_index.items():
    print(word, encoder_embedding_matrix[idx][:5])

In [26]:
#Decoder Embeddings

decoder_embedding_layer = model.get_layer("embedding_3")
decoder_embedding_matrix = decoder_embedding_layer.get_weights()[0]

for word, idx in dec_tokenizer.word_index.items():
    print(word, decoder_embedding_matrix[idx][:5])


<start> [ 0.05496166  0.13549496  0.00179723 -0.12443415 -0.05243227]
<end> [-0.01568825 -0.03688588 -0.00121399 -0.03030608  0.03885898]
je [0.04289194 0.05686763 0.10551131 0.07260018 0.08918607]
mange [ 0.15837738  0.12722886  0.0836602  -0.14829785  0.16506407]
des [ 0.10312185 -0.00284115  0.13784225 -0.04799333  0.00948678]
pommes [ 0.02404149  0.12231556 -0.1576831  -0.13180205  0.02400645]
j [-0.06478216  0.09966506 -0.15847102  0.10550141 -0.15142272]
aime [ 0.05245861 -0.12113839  0.2122246   0.07028405 -0.06319161]
les [ 0.1712255  -0.03404792 -0.21740472  0.1848769   0.10394133]
chiens [ 0.05460045  0.11664888  0.0437142  -0.19585805  0.1466987 ]


In [12]:
import numpy as np

def translate(sentence):
    # Encode
    seq = enc_tokenizer.texts_to_sequences([sentence])
    seq = pad_sequences(seq, maxlen=max_enc_len, padding="post")

    states = encoder_model.predict(seq)

    word = "<start>"
    result = []

    for _ in range(max_dec_len):
        token = dec_tokenizer.texts_to_sequences([[word]])
        token = pad_sequences(token, maxlen=1)

        preds, h, c = decoder_model.predict([token] + states)

        word_id = np.argmax(preds[0, 0])
        word = dec_tokenizer.index_word[word_id]

        if word == "<end>":
            break

        result.append(word)
        states = [h, c]   # 🔑 UPDATE decoder state

    return " ".join(result)


In [14]:
# print(translate("i eat apples"))
print(translate("i eat apples"))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 140ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 181ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step
je mange des pommes
